In [1]:
"""
Use aimsgb to generate grain boundaries, then use 
GPUMD to relax such structures into realistic configurations.
"""
from dotenv import load_dotenv
from aimsgb import GrainBoundary, Grain
from ase.visualize import view
load_dotenv()

s_input = Grain.from_mp_id("mp-149")  # Silicon
UC_A = 2
UC_B = 2

/home/djr2473/.conda/envs/chem/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Retrieving MaterialsDoc documents: 100%|██████████| 1/1 [00:00<00:00, 14074.85it/s]


In [4]:
# In Lortaraprasert and Shiomi 2022, they use:
# 1. Sigma 5 (2 -1 0) [0 0 1]
# 2. Sigma 9 (-1 -2 1) [1 1 0]
# 3. Sigma 9 (2 -2 1) [2 1 2]
# 4. Sigma 3 (1 -1 2) [1 1 0]
# 5. Sigma 5 (3 1 0) [3 1 0]
# 6. Sigma 13 (1 0 0) [1 0 0]
GB_LIST = [
    (5, (2, -1, 0), (0, 0, 1)),
    # (9, (-2, 2, 1), (1, 1, 0)),
    # (9, (-2, 2, 1), (2, 1, 2)),
    # (3, (1, -1, 2), (1, 1, 0)),
    # (5, (3, 1, 0), (3, 1, 0)),
    (13, (1, 0, 0), (1, 0, 0))
]

for (sigma, miller, axis) in GB_LIST:
    gb = GrainBoundary(axis, sigma, miller, s_input, uc_a=UC_A, uc_b=UC_B)
    structure = Grain.stack_grains(gb.grain_a, gb.grain_b, direction=gb.direction)

    view(structure.to_ase_atoms())

# then convert to ASE atoms, write into GPUMD format
# use SW potential with cooling ramp to relax structure (4000K to 25K over 1275ps, simulation step 1fs)
# then a final energy relaxation using L-BFGS
# repeat this process 10 times and get the most energetically stable structure

usage: ase [-h] [--version] [-T]
           {help,info,test,gui,db,run,band-structure,build,dimensionality,eos,ulm,find,nebplot,convert,reciprocal,completion,diff,exec}
           ...
ase: error: TclError: no display name and no $DISPLAY environment variable
To get a full traceback, use: ase -T gui ...


KeyboardInterrupt: 

In [3]:
from aimsgb import GBInformation

print(GBInformation([1, 0, 1], 30).__str__())

Grain boundary information for rotation axis: 101
Show the sigma values up to 30 (Note: * means twist GB, Theta is the rotation angle)
|  Sigma  |  Theta  | GB plane   | CSL matrix   |
|---------+---------+------------+--------------|
|    3    |  70.53  | (1 1 -1)   | 1 -1  1      |
|         |         | (-1 2 1)   | 1  2  0      |
|         |         | (1 0 1)*   | -1  1  1     |
|    9    |  38.94  | (1 -1 -1)  | 1  1  1      |
|         |         | (1 2 -1)   | -1  2  0     |
|         |         | (1 0 1)*   | -1 -1  1     |
|   11    |  50.48  | (1 -3 -1)  | 1  3  1      |
|         |         | (3 2 -3)   | -3  2  0     |
|         |         | (1 0 1)*   | -1 -3  1     |
|   17    |  86.63  | (-2 3 2)   | -2 -3  1     |
|         |         | (-3 -4 3)  | 3 -4  0      |
|         |         | (1 0 1)*   | 2  3  1      |
|   19    |  26.53  | (3 1 -3)   | 3 -1  1      |
|         |         | (-1 6 1)   | 1  6  0      |
|         |         | (1 0 1)*   | -3  1  1     |
|   27    |  31

usage: ase [-h] [--version] [-T]
           {help,info,test,gui,db,run,band-structure,build,dimensionality,eos,ulm,find,nebplot,convert,reciprocal,completion,diff,exec}
           ...
ase: error: TclError: no display name and no $DISPLAY environment variable
To get a full traceback, use: ase -T gui ...
